# Scraping YouTube Comments
**Nama**: Asep Haryana Saputra
**Project**: Sentiment Analysis (YouTube Comments)

This notebook scrapes comments strictly from YouTube to meet the project requirement.

In [11]:
import pandas as pd
from youtube_comment_downloader import YoutubeCommentDownloader
import itertools
import os
import time

VIDEO_URLS = [
    'https://www.youtube.com/watch?v=k9a5LpSfp7c'
]

TARGET_TOTAL = 15000
OUTPUT_FILE = 'dataset_youtube.csv'

In [ ]:
def scrape_multiple_videos(urls, target_total, output_file):
    downloader = YoutubeCommentDownloader()
    
    if os.path.exists(output_file):
        os.remove(output_file)
    
    pd.DataFrame(columns=['cid', 'text', 'time', 'author', 'votes', 'photo', 'heart']).to_csv(output_file, index=False)
    
    total_collected = 0
    per_video_limit = (target_total // len(urls)) + 500

    for url in urls:
        if total_collected >= target_total:
            break
            
        print(f"\nScraping {url}...")
        try:
            comments = downloader.get_comments_from_url(url, sort_by=1)
            
            video_count = 0
            batch_data = []
            
            for comment in itertools.islice(comments, per_video_limit):
                batch_data.append({
                    'cid': comment['cid'],
                    'text': comment['text'],
                    'time': comment['time'],
                    'author': comment['author'],
                    'votes': comment.get('votes', 0),
                    'photo': comment.get('photo', ''),
                    'heart': comment.get('heart', False)
                })
                video_count += 1
                total_collected += 1
                
                if video_count % 100 == 0:
                    # Append to CSV
                    df_batch = pd.DataFrame(batch_data)
                    df_batch.to_csv(output_file, mode='a', header=False, index=False)
                    batch_data = []
                    print(f"Video: {video_count} | Total: {total_collected}", end='\r')
            
            if batch_data:
                df_batch = pd.DataFrame(batch_data)
                df_batch.to_csv(output_file, mode='a', header=False, index=False)
                
            print(f"\nFinished video. Collected: {video_count}")
            
        except Exception as e:
            print(f"\nError scraping {url}: {e}")
            continue
            
    print(f"\nScraping complete. Total rows: {total_collected}")
    
scrape_multiple_videos(VIDEO_URLS, TARGET_TOTAL, OUTPUT_FILE)


Scraping https://www.youtube.com/watch?v=k9a5LpSfp7c...
Video: 15500 | Total: 15500
Finished video. Collected: 15500

Scraping complete. Total rows: 15500


In [ ]:
if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"Loaded {len(df)} rows from {OUTPUT_FILE}")
    print(df.head())

Loaded 15500 rows from dataset_youtube.csv
                          cid  \
0  UgwPwqqKva-HTdnXYTZ4AaABAg   
1  UgxR1u7olr8-eqRcFKF4AaABAg   
2  Ugy9oP4rkZ1mBBE8KZZ4AaABAg   
3  UgwiJeLZmh_Okzt5vA14AaABAg   
4  Ugy3C143yPQfO8gImBR4AaABAg   

                                                text               time  \
0  Apakah ada kejutan lagi dari mas lutfi halimawan?  5 tahun yang lalu   
1                           Mana y orangnya sekarang  4 bulan yang lalu   
2                        nama dia siapa sih gw lupa😅  9 bulan yang lalu   
3         nanti tonton ini ah, sambil main di ODIN4D  1 tahun yang lalu   
4                           Bang, ko ga upload lagii  1 tahun yang lalu   

             author  votes                                              photo  \
0   @RayBuatTrailer  10 rb  https://yt3.ggpht.com/ytc/AIdro_kLZbjoLKXjZu46...   
1   @SumaiyahGendut      0  https://yt3.ggpht.com/ec1JjokDrIS37438GFsv8IS0...   
2  @MamatAlkatiriii      0  https://yt3.ggpht.com/ytc/AIdro_kIQd